In [2]:
import json
import os
import cv2
import numpy as np
from tqdm import tqdm
from shutil import copyfile
from ultralytics import YOLO
from ultralytics.data.converter import convert_coco
from matplotlib import pyplot as plt

In [2]:
# Define paths
DATASET_DIR = r"C:\Users\Bakwowi Junior\Documents\school-documents\4th-semester-SS-26\Computer Vision\project\Krones_dataset"
TRAIN_IMAGES_DIR = os.path.join(DATASET_DIR, "train_images")
TRAIN_ANNOTATIONS_FILE = os.path.join(DATASET_DIR, "train_annotations.json")
FINAL_TRAIN_IMAGES_DIR = os.path.join(DATASET_DIR, "images")
# FINAL_TRAIN_LABELS_DIR = os.path.join(DATASET_DIR, "final_train_labels")


if not os.path.exists(FINAL_TRAIN_IMAGES_DIR):
    os.makedirs(FINAL_TRAIN_IMAGES_DIR)

# if not os.path.exists(FINAL_TRAIN_LABELS_DIR):
#     os.makedirs(FINAL_TRAIN_LABELS_DIR)

print(DATASET_DIR, '\n', TRAIN_IMAGES_DIR, '\n', TRAIN_ANNOTATIONS_FILE, '\n', FINAL_TRAIN_IMAGES_DIR)

print(os.path.exists(DATASET_DIR), os.path.exists(TRAIN_IMAGES_DIR), 
              os.path.exists(TRAIN_ANNOTATIONS_FILE), os.path.exists(FINAL_TRAIN_IMAGES_DIR))


C:\Users\Bakwowi Junior\Documents\school-documents\4th-semester-SS-26\Computer Vision\project\Krones_dataset 
 C:\Users\Bakwowi Junior\Documents\school-documents\4th-semester-SS-26\Computer Vision\project\Krones_dataset\train_images 
 C:\Users\Bakwowi Junior\Documents\school-documents\4th-semester-SS-26\Computer Vision\project\Krones_dataset\train_annotations.json 
 C:\Users\Bakwowi Junior\Documents\school-documents\4th-semester-SS-26\Computer Vision\project\Krones_dataset\images
True False False True


In [3]:
# Preprocessing steps

def apply_clahe(image):
    # Apply Contrast Limited Adaptive Histogram Equalization to enhance image contrast.
    lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)
    limg = cv2.merge((cl, a, b))
    return cv2.cvtColor(limg, cv2.COLOR_LAB2BGR)


def calculate_roi_bounds(roi_annotations):
    # Calculate the bounding box of all annotations in an image.
    
    x_coords = []
    y_coords = []
    
    for ann in roi_annotations:
        x, y, w, h = ann['bbox']
        x_coords.extend([x, x + w])
        y_coords.extend([y, y + h])
    
    return int(min(x_coords)), int(max(x_coords)), int(min(y_coords)), int(max(y_coords))


def compute_crop_region(img_shape, x_min, x_max, y_min, y_max, padding=50):
    # Compute crop region with padding, ensuring it stays within image bounds.
    
    x_offset = max(0, x_min - padding)
    y_offset = max(0, y_min - padding)
    
    crop_w = min(img_shape[1], x_max + padding) - x_offset
    crop_h = min(img_shape[0], y_max + padding) - y_offset
    
    return x_offset, y_offset, crop_w, crop_h


def is_annotation_center_in_crop(bbox, x_offset, y_offset, crop_w, crop_h):
    # Check if the center of a bounding box is within the crop region.
    
    new_x = bbox[0] - x_offset
    new_y = bbox[1] - y_offset
    center_x = new_x + (bbox[2] / 2)
    center_y = new_y + (bbox[3] / 2)
    
    return 0 <= center_x <= crop_w and 0 <= center_y <= crop_h


def adjust_annotation(ann, x_offset, y_offset, ann_id):
    # Adjust annotation coordinates to match cropped image.
    
    old_bbox = ann['bbox']
    new_ann = ann.copy()
    new_ann['bbox'] = [
        old_bbox[0] - x_offset,
        old_bbox[1] - y_offset,
        old_bbox[2],
        old_bbox[3]
    ]
    new_ann['id'] = ann_id
    return new_ann

In [4]:
# Function to detect circles using Hough Transform, 
# crop the image around the detected circle, 
# and adjust annotations accordingly
def crop_image_and_adjust_annotations(json_path, img_dir, img_out_dir, output_dir, padding=50):
    # Crop images based on annotation ROI and adjust annotation coordinates.
    
    with open(json_path, 'r') as f:
        coco_data = json.load(f)

    # Build annotation map for quick lookup
    ann_map = {}
    for ann in tqdm(coco_data['annotations'], desc="Processing annotations", unit="annotation", 
                    ncols=80, bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"):
        ann_map.setdefault(ann['image_id'], []).append(ann)

    new_images = []
    new_annotations = []
    ann_id_counter = 1

    for img_info in tqdm(coco_data['images'], desc="Processing images", unit="image", 
                        ncols=80, bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"):
        img_id = img_info['id']
        filename = img_info['file_name']
        img_path = os.path.join(img_dir, filename)
        img = cv2.imread(img_path)
        
        if img is None:
            print(f"Error: Could not load image at {img_path}. Check the file path!")
            continue

        # Skip images without annotations
        if img_id not in ann_map:
            print(f"Warning: No annotations found for image ID {img_id}. Skipping.")
            continue

        # Calculate ROI bounds from all annotations were the category id is 22 (ROI)
        roi_annotations = [ann for ann in ann_map[img_id] if ann['category_id'] == 22]
        x_min, x_max, y_min, y_max = calculate_roi_bounds(roi_annotations)
        
        # Compute crop region with padding
        x_offset, y_offset, crop_w, crop_h = compute_crop_region(
            img.shape, x_min, x_max, y_min, y_max, padding
        )
        
        # Crop and enhance image
        cropped_img = img[y_offset:y_offset+crop_h, x_offset:x_offset+crop_w]
        cropped_img = apply_clahe(cropped_img)
        cv2.imwrite(os.path.join(img_out_dir, filename), cropped_img)
        
        # Process annotations for this image
        for ann in ann_map[img_id]:
            # Check if annotation center is still within crop region
            if is_annotation_center_in_crop(ann['bbox'], x_offset, y_offset, crop_w, crop_h):
                adjusted_ann = adjust_annotation(ann, x_offset, y_offset, ann_id_counter)
                new_annotations.append(adjusted_ann)
                ann_id_counter += 1
        
        # remove the remaining images after processing the first 30k images
        # if len(new_images) >= 30000:
        #     break

        # Update image metadata
        new_img_info = img_info.copy()
        new_img_info['width'] = crop_w
        new_img_info['height'] = crop_h
        new_images.append(new_img_info)

    # Save adjusted COCO JSON
    output_coco = {
        "info": {
            "year": 2026,
            "version": "1.0",
            "description": "For object detection",
            "date_created": "2026"
        },
        "images": new_images,
        "annotations": new_annotations,
        "categories": coco_data['categories']
        }
    
    with open(os.path.join(output_dir, 'adjusted_annotations.json'), 'w') as f:
        json.dump(output_coco, f)

In [33]:
# call the function to crop images and adjust annotations
crop_image_and_adjust_annotations(
    TRAIN_ANNOTATIONS_FILE,
    TRAIN_IMAGES_DIR,
    FINAL_TRAIN_IMAGES_DIR,
    DATASET_DIR
)

Processing annotations: 100%|█| 134471/134471 [00:00<00:00, 2345028.78annotation
Processing images: 100%|███████████████| 35342/35342 [24:37<00:00, 23.92image/s]


In [6]:
#  Convert annotations in COCO format to YOLO format
# The images and JSON file should be in the same parent folder
def convert_annotations_coco_to_yolo_format(coco_json, output_dir):
    return convert_coco(
        labels_dir=coco_json,
        save_dir=output_dir,
        use_segments=False,
        cls91to80=False
    )

adjusted_annotations_path = os.path.join(DATASET_DIR, 'adjusted_annotations.json')

convert_annotations_coco_to_yolo_format(
    coco_json=DATASET_DIR,  # Pass the directory containing the JSON file
    output_dir=DATASET_DIR
)

Annotations C:\Users\Bakwowi Junior\Documents\school-documents\4th-semester-SS-26\Computer Vision\project\Krones_dataset\adjusted_annotations.json: 100% ━━━━━━━━━━━━ 35342/35342 1.4Kit/s 25.2s<0.0s
COCO data converted successfully.
Results saved to C:\Users\Bakwowi Junior\Documents\school-documents\4th-semester-SS-26\Computer Vision\project\Krones_dataset-2


In [7]:
# count the number of txt files in the labels directory
# labels_dir = r"C:\Users\Bakwowi Junior\Documents\school-documents\4th-semester-SS-26\Computer Vision\project\Krones_dataset-2\labels\adjusted_annotations"
labels_dir = r"C:\Users\Bakwowi Junior\Documents\school-documents\4th-semester-SS-26\Computer Vision\project\Krones_training_dataset-cv\labels\adjusted_annotations"
txt_files = [f for f in os.listdir(labels_dir) if f.endswith('.txt')]
print(f"Number of .txt files in {labels_dir}: {len(txt_files)}")

Number of .txt files in C:\Users\Bakwowi Junior\Documents\school-documents\4th-semester-SS-26\Computer Vision\project\Krones_training_dataset-cv\labels\adjusted_annotations: 35342


In [6]:
# crop the images in the test_images directory
def get_inference_crop(full_img, offset=50):
    gray = cv2.medianBlur(cv2.cvtColor(full_img, cv2.COLOR_BGR2GRAY), 5)
    circles = cv2.HoughCircles(gray, cv2.HOUGH_GRADIENT, 1.2, 100, 
                               param1=100, param2=60, minRadius=200, maxRadius=300)
    
    if circles is not None:
        # Strategy A: Use the Dynamic Circle
        # print("Circle detected-cropping around it.", circles)
        x, y, r = np.round(circles[0, 0]).astype("int")

        return full_img[y-r-offset : y+r+offset, x-r-offset : x+r+offset], (x-r-offset, y-r-offset)
    else:
        # Strategy B: Fallback to the "Safe" ROI
        # Use the fixed coordinates you used when you removed Hough Circles earlier
        x, y, w, h = [320, 200, 612, 612] 
        # print("No circle detected, using fallback crop.", (x, y, w, h))
        return full_img[y:y+h, x:x+w], (x, y)

def apply_clahe_to_crop(cropped_img):
    # Apply CLAHE to the cropped image
    # print("Applying CLAHE to cropped image.", cropped_img)
    lab = cv2.cvtColor(cropped_img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)
    limg = cv2.merge((cl, a, b))
    return cv2.cvtColor(limg, cv2.COLOR_LAB2BGR)

test_dir = r"C:\Users\Bakwowi Junior\Documents\school-documents\4th-semester-SS-26\Computer Vision\project\Krones_dataset_cv\test_images"

for img_file in tqdm(os.listdir(TEST_IMAGES_DIR), desc="Cropping test images", unit="image", 
                     ncols=80, bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"):
    if img_file.endswith('.png'):
        img_path = os.path.join(TEST_IMAGES_DIR, img_file)
        full_img = cv2.imread(img_path)
        cropped_img, (x_offset, y_offset) = get_inference_crop(full_img)
        try:
            clahe_img = apply_clahe_to_crop(cropped_img)
            cv2.imwrite(os.path.join(test_dir, img_file), clahe_img)
        except Exception as e:
            print(f"Error processing {img_file}: {e}", cropped_img)

# img = cv2.imread(os.path.join(TEST_IMAGES_DIR, "0d64352f-0cf6-498f-949a-6a8a6d8c8b6d_000000003648.png"))
# cropped_img, (x_offset, y_offset) = get_inference_crop(img)
# # print("Cropped image shape:", cropped_img)

# clahe_img = apply_clahe_to_crop(cropped_img)
# plt.imshow(cv2.cvtColor(clahe_img, cv2.COLOR_BGR2RGB))
# plt.axis('off')
# plt.show()


Cropping test images: 100%|██████████████| 4418/4418 [07:50<00:00,  9.38image/s]


In [ ]:
# # open one of the test images
# img = cv2.imread(r"C:\Users\Bakwowi Junior\Documents\school-documents\4th-semester-SS-26\Computer Vision\project\Krones_dataset\test_images\0d64352f-0cf6-498f-949a-6a8a6d8c8b6d_000000003648.png")
# print("Original image shape:", img.shape)
# plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
# plt.title("Original Image")

In [6]:
# import os
import random
import shutil

# Paths
image_dir = r'C:\Users\Bakwowi Junior\Documents\school-documents\4th-semester-SS-26\Computer Vision\project\Krones_training_dataset-cv\train\images'
label_dir = r'C:\Users\Bakwowi Junior\Documents\school-documents\4th-semester-SS-26\Computer Vision\project\Krones_training_dataset-cv\train\labels'
val_img_dir = r'C:\Users\Bakwowi Junior\Documents\school-documents\4th-semester-SS-26\Computer Vision\project\Krones_training_dataset-cv\val\images'
val_label_dir = r'C:\Users\Bakwowi Junior\Documents\school-documents\4th-semester-SS-26\Computer Vision\project\Krones_training_dataset-cv\val\labels'

# Create folders
os.makedirs(val_img_dir, exist_ok=True)
os.makedirs(val_label_dir, exist_ok=True)

# Get all image files
images = [f for f in os.listdir(image_dir) if f.endswith('.png')]
random.shuffle(images)

# Calculate 20% for validation
val_count = int(len(images) * 0.20)
val_samples = images[:val_count]

for filename in tqdm(val_samples, desc="Moving validation samples", unit="file", 
                 ncols=80, bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"):
    # Move Image
    shutil.move(os.path.join(image_dir, filename), os.path.join(val_img_dir, filename))
    
    # Move matching Label
    label_file = filename.replace('.png', '.txt')
    if os.path.exists(os.path.join(label_dir, label_file)):
        shutil.move(os.path.join(label_dir, label_file), os.path.join(val_label_dir, label_file))

print(f"Moved {val_count} images and labels to validation set.")

Moving validation samples:   0%|                     | 0/7068 [00:00<?, ?file/s]

Moving validation samples: 100%|█████████| 7068/7068 [00:10<00:00, 665.74file/s]

Moved 7068 images and labels to validation set.


In [ ]:
# Define the YOLO model and evolve hyperparameters
if __name__ == "__main__":
    yaml_data = r"C:\Users\Bakwowi Junior\Documents\school-documents\4th-semester-SS-26\Computer Vision\project\Krones_training_dataset-cv\image_data.yaml"
    model = YOLO('yolo26n.pt')
    model.tune(
        data=yaml_data,
        epochs=10,        # Short bursts to test settings
        iterations=10,   # 100 different 'mutations' of hyperparameters
        imgsz=640, 
        device=0,
        workers=4,
        # Specific industrial priors:
        scale=0.5,        # Focus on small objects
        fliplr=0.5,       # Bottles rotate, so symmetry matters
        mosaic=1.0        # Mandatory for detecting multiple small spots
    )

    # Final Training
    model.train(
        data=yaml_data,
        epochs=50,
        imgsz=640,
        batch=32,
        patience=25,
        # High-speed industrial augmentations
        mixup=0.2, 
        mosaic=1.0, 
        degrees=180.0 # Bottles spin on the line
    )

In [6]:
# predict on test images and save results
test_dir = r"C:\Users\Bakwowi Junior\Documents\school-documents\4th-semester-SS-26\Computer Vision\project\Krones_dataset_cv\test_images"
results_dir = r"C:\Users\Bakwowi Junior\Documents\school-documents\4th-semester-SS-26\Computer Vision\project\Krones_dataset_cv\results"
os.makedirs(results_dir, exist_ok=True)
model = YOLO("final_model.pt")
for img_file in tqdm(os.listdir(test_dir), desc="Predicting on test images", unit="image", 
                     ncols=80, bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"):
    if img_file.endswith('.png'):
        img_path = os.path.join(test_dir, img_file)
        results = model(img_path)
        results[0].save(filename=os.path.join(results_dir, img_file))



Predicting on test images:   0%|                    | 0/4418 [00:00<?, ?image/s]


image 1/1 C:\Users\Bakwowi Junior\Documents\school-documents\4th-semester-SS-26\Computer Vision\project\Krones_dataset_cv\test_images\00050d96-644f-4f14-89d5-eb9cd1c05f55_000000001443.png: 512x640 1 Roi, 59.8ms
Speed: 1.8ms preprocess, 59.8ms inference, 0.3ms postprocess per image at shape (1, 3, 512, 640)


Predicting on test images:   0%|            | 1/4418 [00:00<12:34,  5.85image/s]


image 1/1 C:\Users\Bakwowi Junior\Documents\school-documents\4th-semester-SS-26\Computer Vision\project\Krones_dataset_cv\test_images\000afea1-bb78-44e4-ad6c-6012654e01f8_000000001316.png: 512x640 1 Roi, 56.3ms
Speed: 1.7ms preprocess, 56.3ms inference, 0.2ms postprocess per image at shape (1, 3, 512, 640)


Predicting on test images:   0%|            | 2/4418 [00:00<11:02,  6.67image/s]

...

Predicting on test images:   1%|▏          | 55/4418 [00:07<10:36,  6.85image/s]


image 1/1 C:\Users\Bakwowi Junior\Documents\school-documents\4th-semester-SS-26\Computer Vision\project\Krones_dataset_cv\test_images\0308e9e8-8607-4e54-9ff9-9e445668e0d6_000000001138.png: 512x640 1 Roi, 57.8ms


Predicting on test images:   1%|▏          | 55/4418 [00:07<10:24,  6.98image/s]

Speed: 1.6ms preprocess, 57.8ms inference, 0.3ms postprocess per image at shape (1, 3, 512, 640)


KeyboardInterrupt: 

In [ ]:
# Mapping Thresholds (Class Name -> Pixel Area Threshold)
CONDITIONALLY_FAULTY = {
    "Air bubble": {"single": 500, "total": 1200}, # Example: 3 bubbles of 450px = Reject
    "Chip": {"single": 200, "total": 400},
    "Contamination light": {"single": 180, "total": 500},
    "Glass imperfection": {"single": 100, "total": 300},
    "Scuffing": {"single": 75000, "total": 100000},
    "Scuffing heavy": {"single": 1200, "total": 2500}
}

ALWAYS_FAULTY = [
    "Break / Crack", "Circlip", "Contamination dark", "Crown cap", 
    "Foil / Semitransparent", "Foreign object - manual cleaning", 
    "Foreign object - washing machine", "Glass shard", "Insect", 
    "Label", "Liquid", "Mold", "No base visible", "Paint residue", 
    "Straw", "Yeast residue"
]

def final_decision(results):
    detections = results[0].boxes
    if len(detections) == 0:
        return 0, "Clear"

    # Trackers for cumulative area per class
    class_area_totals = {} 
    
    for det in detections:
        label = model.names[int(det.cls)]
        area = float(det.xywh[0][2] * det.xywh[0][3])

        # 1. Immediate Rejection (Faulty)
        if label in ALWAYS_FAULTY:
            return 1, f"Critical Failure: {label}"

        # 2. Check Individual & Cumulative (Conditionally Faulty)
        if label in CONDITIONALLY_FAULTY:
            limits = CONDITIONALLY_FAULTY[label]
            
            # Check single object threshold
            if area > limits["single"]:
                return 1, f"Reject: Single {label} too large ({area:.0f}px)"
            
            # Update running total for this class
            class_area_totals[label] = class_area_totals.get(label, 0) + area
            
            # Check cumulative threshold
            if class_area_totals[label] > limits["total"]:
                return 1, f"Reject: Cumulative {label} exceeded ({class_area_totals[label]:.0f}px)"

    return 0, "Pass"

In [ ]:
#Load the trained model
# model = YOLO('models/example.pt')

# Run validation
# This generates the Confusion Matrix, P-curve, R-curve, and F1-curve automatically
# results = model.val(data='image_data.yaml', imgsz=640, split='val')

# # Access specific metrics programmatically
# print(f"Mean Precision: {results.results_dict['metrics/precision(B)']:.4f}")
# print(f"Mean Recall: {results.results_dict['metrics/recall(B)']:.4f}")
# print(f"Mean F1-Score: {results.results_dict['metrics/f1(B)']:.4f}")
# print(f"mAP@50 (Accuracy): {results.results_dict['metrics/mAP50(B)']:.4f}")

In [24]:
# count the number of images in the train_images directory
train_images_count = len([f for f in os.listdir(TRAIN_IMAGES_DIR) if f.endswith('.png')])
print(f"Number of images in {TRAIN_IMAGES_DIR}: {train_images_count}")

Number of images in C:\Users\Bakwowi Junior\Documents\school-documents\4th-semester-SS-26\Computer Vision\project\Krones_dataset\train_images: 35342


In [23]:
# Visualize the dataset with FiftyOne
import fiftyone as fo
import json
import os

# # Path to dataset
# dataset_dir = r"C:\Users\Bakwowi Junior\Documents\school-documents\4th-semester-SS-26\Computer Vision\project\Krones_dataset_cv"
# images_dir = os.path.join(dataset_dir, 'images')
# coco_path = os.path.join(dataset_dir, 'adjusted_annotations.json')

# # Load COCO data manually
# with open(coco_path, 'r') as f:
#     coco_data = json.load(f)

# # Create a mapping of category IDs to names
# category_map = {cat['id']: cat['name'] for cat in coco_data['categories']}

dataset_name = "Krones_dataset_cv"

# print(fo.list_datasets())
# # Delete existing dataset if it exists
# if dataset_name in fo.list_datasets():
#     print(f"Deleting existing dataset '{dataset_name}'...")
#     fo.delete_dataset(dataset_name)

# print(f"Creating new dataset '{dataset_name}'...")

# # Create dataset
# dataset = fo.Dataset(name=dataset_name)

# # Add samples with detections
# image_ann_map = {}
# for ann in tqdm(coco_data['annotations'], desc="Mapping annotations to images", unit="annotation",
#                 ncols=80, bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"):
#     img_id = ann['image_id']
#     if img_id not in image_ann_map:
#         image_ann_map[img_id] = []
#     image_ann_map[img_id].append(ann)

# for img_info in tqdm(coco_data['images'], desc="Loading images", unit="image",
#                      ncols=80, bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"):
#     img_id = img_info['id']
#     img_path = os.path.join(images_dir, img_info['file_name'])
    
#     if not os.path.exists(img_path):
#         print(f"Warning: Image not found: {img_path}")
#         continue
    
#     sample = fo.Sample(filepath=img_path)
    
#     # Add detections for this image
#     detections = []
#     if img_id in image_ann_map:
#         for ann in image_ann_map[img_id]:
#             bbox = ann['bbox']  # [x, y, width, height] in image coordinates
#             width = img_info['width']
#             height = img_info['height']
            
#             # Convert to normalized coordinates [top_left_x, top_left_y, width, height]
#             norm_bbox = [
#                 bbox[0] / width,
#                 bbox[1] / height,
#                 bbox[2] / width,
#                 bbox[3] / height
#             ]
            
#             label = category_map.get(ann['category_id'], 'unknown')
#             detection = fo.Detection(label=label, bounding_box=norm_bbox)
#             detections.append(detection)
    
#     sample['detections'] = fo.Detections(detections=detections)
#     dataset.add_sample(sample)

# print(f"Loaded {len(dataset)} samples into FiftyOne")

print(f"Loading existing dataset '{dataset_name}'...")
dataset = fo.load_dataset(dataset_name)
print(f"Dataset loaded with {len(dataset)} samples")

# Launch FiftyOne App for interactive visualization
print("Launching FiftyOne app...")
try:
    # Try different ports if 5151 is busy
    ports_to_try = [5151, 5152, 5153, 5154, 5155]
    session = None

    for port in ports_to_try:
        try:
            print(f"Trying port {port}...")
            session = fo.launch_app(dataset, port=port, auto=False)  # auto=False prevents auto-opening browser
            print(f"Successfully launched on port {port}")
            print(f"Open your browser and go to: http://localhost:{port}")
            break
        except Exception as e:
            print(f"Port {port} failed: {e}")
            continue

    if session is None:
        print("Could not launch FiftyOne app on any port. Try running:")
        print("fo.launch_app(dataset)")
        print("Or check if FiftyOne is properly installed: pip install fiftyone[desktop]")
    else:
        # Don't use session.wait() as it can cause connection issues
        print("FiftyOne app is running. Press Ctrl+C in terminal to stop when done.")

except Exception as e:
    print(f"Error launching FiftyOne: {e}")
    print("Try running the following in a separate cell:")
    print("fo.launch_app(dataset)")

Loading existing dataset 'Krones_dataset_cv'...
Dataset loaded with 30000 samples
Launching FiftyOne app...
Trying port 5151...
Session launched. Run `session.show()` to open the App in a cell output.
Successfully launched on port 5151
Open your browser and go to: http://localhost:5151
FiftyOne app is running. Press Ctrl+C in terminal to stop when done.


In [ ]:
def is_image_good(img):
    """Checks if the image is sharp enough to process."""
    if img is None:
        return False
    
    # Convert to grayscale for analysis
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Laplacian Variance: Higher = Sharper
    score = cv2.Laplacian(gray, cv2.CV_64F).var()
    
    return score > 60  # Threshold for sharpness (adjust as needed)

def is_image_useful(img, brightness_min=5, contrast_min=10):
    """
    Checks if an image has enough 'life' to be processed.
    - brightness_min: Filters out absolute black frames.
    - contrast_min: Filters out flat, featureless grey/blur frames.
    """
    if img is None:
        return False
    
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Calculate stats
    mean_brightness = np.mean(gray)
    contrast = np.std(gray) # Standard Deviation
    
    # If it's pitch black (mean < 5), skip it.
    # If it's a flat, featureless color (std < 10), skip it.
    if mean_brightness < brightness_min or contrast < contrast_min:
        return False
        
    return True

# get the first 100 images from the test set and check if they are good for processing
# for file in list(os.listdir(TEST_IMAGES_DIR))[:100]:
#     img_path = os.path.join(TEST_IMAGES_DIR, file)
#     img = cv2.imread(img_path)
    
#     if not is_image_useful(img):
#         print(f"Image {file} is too blurry. Consider removing it from the dataset.")
#     else:
#         print(f"Image {file} is good for processing.")

plt.imshow(cv2.imread(os.path.join(TEST_IMAGES_DIR, "008104cd-8e3a-44b7-8354-008ec2e20d2f_000000001491.png")), cmap='gray')

In [ ]:
# Visualize class distribution for the first 32k images using FiftyOne and Plotly
import fiftyone as fo
import plotly.graph_objects as go
import plotly.express as px
from collections import Counter
import json

# Load COCO annotations
coco_path = os.path.join(DATASET_DIR, 'train_annotations.json')
with open(coco_path, 'r') as f:
    coco_data = json.load(f)

# Create category mapping
category_id_to_name = {cat['id']: cat['name'] for cat in coco_data['categories']}

# Count class distribution from the first 32k images
class_counts = Counter()
image_count = 0
max_images = min(30000, len(coco_data['images']))

# Create a set of first 32k image IDs
first_32k_image_ids = set(img['id'] for img in coco_data['images'][:max_images])

# Count detections for classes in first 32k images
for ann in coco_data['annotations']:
    if ann['image_id'] in first_32k_image_ids:
        class_name = category_id_to_name.get(ann['category_id'], 'Unknown')
        class_counts[class_name] += 1

# Sort by count (descending)
sorted_classes = sorted(class_counts.items(), key=lambda x: x[1], reverse=True)
classes = [item[0] for item in sorted_classes]
counts = [item[1] for item in sorted_classes]

# Create Plotly horizontal bar chart
fig = go.Figure(data=[
    go.Bar(
        y=classes,
        x=counts,
        orientation='h',
        marker=dict(
            color=counts,
            colorscale='Viridis',
            showscale=True,
            colorbar=dict(title="Count")
        ),
        text=counts,
        textposition='auto',
        hovertemplate='<b>%{y}</b><br>Detections: %{x}<extra></extra>'
    )
])

fig.update_layout(
    title=f'Class Distribution - First {max_images} Images',
    xaxis_title='Number of Detections',
    yaxis_title='Class',
    height=600,
    width=1000,
    font=dict(size=11),
    plot_bgcolor='rgba(240, 240, 240, 0.5)',
    hovermode='closest'
)

fig.show()

# Print summary statistics
print(f"\n{'='*60}")
print(f"Class Distribution Summary (First {max_images} Images)")
print(f"{'='*60}")
print(f"Total detections: {sum(counts)}")
print(f"Total unique classes: {len(classes)}")
print(f"\nDetailed Breakdown:")
print(f"{'-'*60}")
for cls, count in sorted_classes:
    percentage = (count / sum(counts)) * 100
    print(f"  {cls:<40} {count:>6} ({percentage:>5.2f}%)")
print(f"{'-'*60}")

In [ ]:
# import plotly plot 
plot = os

In [ ]:
sample_img = cv2.imread('sample3.png')


# apply hough circle transform to detect and draw circles in the image
def detect_circles(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        # Median blur is better for brown glass noise than Gaussian
    # gray = cv2.medianBlur(gray, 9) 
    
    circles = cv2.HoughCircles(gray, cv2.HOUGH_GRADIENT, dp=1.2, minDist=300,
                                param1=50, param2=40, minRadius=180, maxRadius=300)
    return circles


circles = np.uint16(np.around(detect_circles(sample_img)))
x, y, r = circles[0, 0]
clipped_img = sample_img[max(0, y-r-20):y+r+20, max(0, x-r-20):x+r+20]
plt.imshow(cv2.cvtColor(clipped_img, cv2.COLOR_BGR2RGB))

# # detect circles in the final image
# circles = detect_circles(sample_img)

# # draw the circles on the image
# if circles is not None:
#     circles = np.uint16(np.around(circles))
#     for i in circles[0, :]:
#         cv2.circle(sample_img, (i[0], i[1]), i[2], (0, 255, 0), 2)
#         cv2.circle(sample_img, (i[0], i[1]), 2, (0, 0, 255), 3)


# # apply clahe to the image
# lab = cv2.cvtColor(sample_img, cv2.COLOR_BGR2LAB)
# l, a, b = cv2.split(lab)
# clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
# cl = clahe.apply(l)
# limg = cv2.merge((cl, a, b))
# final_img = cv2.cvtColor(limg, cv2.COLOR_LAB2BGR)


# # save the final image
# cv2.imwrite('final_sample3.png', final_img)
# cv2.imwrite('updated_sample3.png', sample_img)

# # display the original and final images
# plt.subplot(1, 2, 1)
# plt.imshow(cv2.cvtColor(sample_img, cv2.COLOR_BGR2RGB))
# plt.title('Original Image')
# plt.axis('off')

# plt.subplot(1, 2, 2)
# plt.imshow(cv2.cvtColor(final_img, cv2.COLOR_BGR2RGB))
# plt.title('CLAHE Image')
# plt.axis('off')
# plt.show()

In [ ]:
# Visualize the dataset with fiftyone
